# Azure ML Iris Endpoint Deployment

This notebook trains an Iris model, registers it, deploys it to a managed online endpoint, and tests scoring.

In [ ]:
!pip install azure-ai-ml azure-identity scikit-learn pandas joblib

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
    CodeConfiguration
)

import joblib
import os

In [ ]:
subscription_id = "1de469b2-53cb-47f9-a520-b9a5dcd554d3"
resource_group = "anotherserver"
workspace_name = "randaao-server"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name
)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

In [ ]:
joblib.dump(model, "model.joblib")

registered_model = ml_client.models.create_or_update(
    Model(
        name="iris-model",
        path="model.joblib"
    )
)

In [ ]:
%%writefile score.py
import json
import joblib
import numpy as np
import os

def init():
    global model
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model.joblib")
    model = joblib.load(model_path)

def run(data):
    try:
        if isinstance(data, str):
            data = json.loads(data)

        inputs = np.array(data["data"])
        preds = model.predict(inputs)
        return preds.tolist()

    except Exception as e:
        return {"error": str(e)}

In [ ]:
%%writefile conda.yml
name: sklearn-env
channels:
  - defaults
dependencies:
  - python=3.8
  - pip
  - pip:
      - scikit-learn
      - joblib
      - numpy
      - azureml-inference-server-http

In [ ]:
env = Environment(
    name="sklearn-env-final",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file="conda.yml"
)

ml_client.environments.create_or_update(env)

In [ ]:
endpoint = ManagedOnlineEndpoint(
    name="iris-endpoint-2345",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="iris-endpoint-2345",
    model=registered_model,
    environment=env,
    code_configuration=CodeConfiguration(
        code="./",
        scoring_script="score.py"
    ),
    instance_type="Standard_F2s_v2",
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

In [ ]:
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [ ]:
endpoint = ml_client.online_endpoints.get("iris-endpoint-2345")
keys = ml_client.online_endpoints.get_keys("iris-endpoint-2345")

print("Scoring URI:", endpoint.scoring_uri)
print("Key:", keys.primary_key)

In [ ]:
import requests
import json

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {keys.primary_key}"
}

data = {
    "data": [
        [5.1, 3.5, 1.4, 0.2]
    ]
}

response = requests.post(endpoint.scoring_uri, headers=headers, data=json.dumps(data))
print(response.json())